<a href="https://colab.research.google.com/github/gnperdue/SaturdayMorningQuantum/blob/main/QuantumComputing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum Computing in 2 Hours*

*Actually learning programming in a short period of time is more challenging than you might think...

<img src="images/1fl5yi6bbpzz.png" height=500>

**Notebooks like this one have a lot of functionality, but the main things you need to know are how to run a "cell" and how to edit one. You can just click on a cell to get an editor, and use "shift+return" to run the cell (either code or markup).**

In [ ]:
#@title "Software Setup [Run this cell with Shift+Enter!]"
try:
    import numpy as np
    import qiskit
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator
    import cirq
    import stim
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install",
         "qiskit", "qiskit-aer", "cirq-core", "stim", "-q"]
    )
    import numpy as np
    import qiskit
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator
    import cirq
    import stim

print(f"NumPy {np.__version__}")
print(f"Qiskit {qiskit.__version__}")
print(f"Cirq {cirq.__version__}")
print(f"Stim {stim.__version__}")

This notebook was last tested with NumPy 2.4.3 and Qiskit 2.3.1.

We will also use some code and functions [originally written by Dr. C. Moran](https://www.frontiersin.org/articles/10.3389/fphy.2018.00069/full) for qubit visualization.

In [ ]:
#@title "NumPy Qubits and Bloch sphere plotting functions [Run me!]"
zero_qubit=np.matrix('1; 0')
one_qubit=np.matrix('0; 1')
plus_qubit=1/np.sqrt(2)*np.matrix('1; 1')
minus_qubit=1/np.sqrt(2)*np.matrix('1; -1')
clockwisearrow_qubit=1/np.sqrt(2)*np.matrix([[1],[complex(0,1)]])
counterclockwisearrow_qubit=1/np.sqrt(2)*np.matrix([[1],[-complex(0,1)]])

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
def get_bloch_coordinates(qubit):
    def get_x_bloch(qubit):
        qubit_x_basis=1./np.sqrt(2)*np.matrix('1 1; 1 -1')*qubit
        prob_zero_qubit=(qubit_x_basis.item(0)*qubit_x_basis.item(0).conjugate()).real
        prob_one_qubit=(qubit_x_basis.item(1)*qubit_x_basis.item(1).conjugate()).real
        return prob_zero_qubit-prob_one_qubit

    def get_y_bloch(qubit):
        qubit_y_basis=1./np.sqrt(2)*np.matrix('1 1; 1 -1')*np.matrix([[1,0],[0,-complex(0,1)]])*qubit
        prob_zero_qubit=(qubit_y_basis.item(0)*qubit_y_basis.item(0).conjugate()).real
        prob_one_qubit=(qubit_y_basis.item(1)*qubit_y_basis.item(1).conjugate()).real
        return prob_zero_qubit-prob_one_qubit

    def get_z_bloch(qubit):
        qubit_z_basis=qubit
        prob_zero_qubit=(qubit_z_basis.item(0)*qubit_z_basis.item(0).conjugate()).real
        prob_one_qubit=(qubit_z_basis.item(1)*qubit_z_basis.item(1).conjugate()).real
        return prob_zero_qubit-prob_one_qubit
    return (get_x_bloch(qubit),get_y_bloch(qubit),get_z_bloch(qubit))

def plot_bloch(qubit,color='b',ax=None):
    if not ax:
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='3d')

        # draw sphere
        u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
        x = np.cos(u)*np.sin(v)
        y = np.sin(u)*np.sin(v)
        z = np.cos(v)
        ax.plot_wireframe(x, y, z, color="k",alpha=.1)
        ax.grid(False)

    coordinates=get_bloch_coordinates(qubit)
    ax.quiver([0],[0],[0],[coordinates[0]],[coordinates[1]],[coordinates[2]],length=1,color=color,arrow_length_ratio=0.3)
    ax.set_xlim([-1,1])
    ax.set_ylim([-1,1])
    ax.set_zlim([-1,1])
    ax.set_xlabel('x: |"-"> to |"+">')
    ax.set_ylabel('y: |"↺"> to |"↻">')
    ax.set_zlabel('z: |"1"> to |"0">')
    ax.view_init(azim=20)
    return ax

i_ = complex(0, 1)
H = 1. / np.sqrt(2) * np.matrix('1 1; 1 -1')
X = np.matrix('0 1; 1 0')
Y = np.matrix([[0, -i_], [i_, 0]])
Z = np.matrix([[1, 0], [0, -1]])

def pretty_print_number(amnt):
    the_string = locale.format_string("%12.0f", amnt, grouping=True)
    print("{}".format(the_string))

# other libraries
import locale
import pprint
import scipy.optimize

## Introduction and front matter

This is a [Jupyter](https://jupyter.org/) notebook. It mixes formatted text, graphics, and executable code in one document. You can run it locally with [JupyterLab](https://jupyterlab.readthedocs.io/) or in the cloud with [Google Colab](https://colab.research.google.com) (free with a Google account).

We are using [Python](https://www.python.org/). If you're new to Python, the [official beginner's guide](https://wiki.python.org/moin/BeginnersGuide) is great. Don't worry if you've never seen Python code before — it is easy to read.

### Outline

0. Brief aside on entanglement and superposition.
1. Computation is tied to _physical systems_.
2. "Classical" computation with the _circuit model_.
3. Limitations of the classical computing model.
4. Distinguishing features of _quantum computation_.
5. The quantum circuit model — lines and gates.
6. Quantum error correction — from repetition codes to the surface code.

## Measurements, Superposition, and Entanglement

Quantum computing draws its power from two phenomena unique to quantum mechanics: **superposition** and **entanglement**. To understand them, we first need to see how **measurement** works at the quantum scale.

### Two-level Systems

#### Classical Example: A coin

A coin is a classical two-level system: it can be either Heads or Tails. It is always one and only one of those options. This can be abstracted one level higher - as a **bit**. Bits can be either 0 or 1. We will assign 0 to Heads on our coin, and 1 to Tails.

<img src="images/ht01.png">

We could use an array of coins to encode a number in binary. TTHTT would be 0b11011 = 27. (See below for more on binary.)

<img src="images/tthtt.png">

#### Quantum Example: A _Spin_

Our quantum two-level system is a **spin**. Spin is an intrinsic property of quantum particles — think of it as built-in angular momentum. Picturing a tiny spinning top is _not_ correct, but it is a useful mental model.

We'll work with spin-1/2 particles, which have exactly two levels: `Up` and `Down`. Map 0 to one and 1 to the other, and we have a **qubit** — a *quantum* bit.

<img src="images/christophe-hautier-902vnYeoWS4-unsplash.jpg" height=400>

Photo by [Christophe Hautier](https://unsplash.com/@hautier)

But if our spin direction is based on angular momentum, can’t it be at some arbitrary angle? Why is this a two-level system? The oddness of quantum mechanics - including the word quantum - comes from these observations about two-level systems!

### Measurement

Suppose we have a device for measuring the **spin** of a quantum particle. Our device measures spin **along a direction**. We’ll call the “up” direction `z`. To the right is `x`, and `y` points “into the page.” What do we measure if we put our measurement axis along `z`?

<img src="images/spin.png" height=400>

**We always measure `Up` or `Down` — never a fractional value like `0.5 Up`.** A collection of random spins gives a roughly even mix of the two. And once we measure a particular spin, repeating the measurement gives the same result every time.

<img src="images/spin2.png" height=400>

Suppose we have easy access to a large number of spins that are definitely either `Up` or `Down` along `z`. Let’s look at a bunch of `Up`s, and measure along `x`.

What do you expect we will find?

<img src="images/spin3.png" height=400>

It turns out we again measure only either “`Right`” (`+x`) or “`Left`” (`-x`) and never anything like `0.5 Right`. We will find an even mix of the two. Why is this? What is going on?

Real insight comes when we measure these spins along the `z` axis again. Suppose we take a lot of `Up`s, measure the `x`, sort them into `Right` and `Left`, and measure those. What do you think we find?

<img src="images/spin4.png" height=400>

For both groups we find an even mix of `Up` and `Down` along `z`! But again, once we have measured the spin along an axis, we can come back and measure it again on that axis and always get the same thing.

This is not how things work classically. Imagine we had some dice sitting on a table and looked at the faces from above, and then from the side… would the faces change when we looked again from above?

What we’re learning here is that **measurement is not a gentle act at the quantum scale.** Quantum particles are disturbed by measurement. We’re also learning the results are *quantized.* (The fact that measurement along `x` disturbs subsequent measurements along `z` is a manifestation of the famous [uncertainty principle](https://en.wikipedia.org/wiki/Uncertainty_principle).)

### Superposition

One way to understand what is happening here is to use superposition. Quantum particles may exist in multiple states simultaneously!

Suppose we have a spin and measure along `z`, and find it is `Up`. But… our choice of the “`Up`” direction was arbitrary. We could have chosen any direction. Suppose we had started with `x`. We would have found `Left` or `Right`. But, and this is the important thing - `Up-Down` is at 90 degrees to `Left-Right`. Being **fully** `Up` or `Down` is equivalent to being in a 50-50 **superposition** of `Left + Right`!

There is an important, subtle difference between the way those things add up for comparing `Up` and `Down`, but let’s set that aside for the moment. Our `Up` is actually something like

$$
|U\rangle = \frac{1}{\sqrt{2}} \left( |L\rangle + |R\rangle \right).
$$

It is a *superposition of states!* (Note our funny-looking notation - this is called "bracket notation".)

<img src="images/spin-xz.png" height=400>

If we measure along the `x` axis, we have a 50-50 chance of measuring `Left` or `Right`. **Measurement collapses the superposition!** But, say we have `Right` - that is a superposition of `Up` and `Down` now! That is why when we return to measure along `z`, we get an even mix of `Up` and `Down` (an even mix over many measurements of independent spins).

One of the most famous superpositions of all time...

<img src="images/blog_schrodinger1.jpg">

What happens if we had chosen, instead of `x`, some other angle that wasn’t 90 degrees from `z`?

We again find a superposition, but this time the amplitudes are proportional to trigonometric functions of the angle - the mix won’t be even. **The probability of measuring a value is equal to the modulus squared of the amplitude.** So, in other words, the probability of finding a specific value changes, but we still only ever find one value or the other.

Probabilities are real numbers constrained between zero and one. We always measure _something_ - the sums of the squares **must always be 1**. If we have

$$
|U\rangle = \cos \theta |L\rangle + \sin \theta |R\rangle.
$$

this relation holds because $\cos^2 \theta + \sin^2 \theta = 1$ for any value of $\theta$.

There is an important point hidden here --- because we only ever measure amplitudes _squared_, amplitudes can be negative. They may even be _complex_, which means they carry imaginary terms. We write complex numbers like $a + i \times b$, where $a$ and $b$ are ordinary numbers, and $i$ is a special constant with the property that $i^2 = -1$.

Note: when we compute the modulus of a complex number we need to follow a special rule:

$$
\left| a + i \times b \right|^2 = (a + i \times b) \times (a - i \times b) = a^2 + b^2
$$

By convention we will write complex numbers with Greek letters. Note that if we write a state like:

$$
|U\rangle = \alpha |L\rangle + \beta |R\rangle
$$

this is really like writing

$$
|U\rangle = (a + i \times b) |L\rangle + (c + i \times d) |R\rangle
$$

with constraints on the relationship between $a$, $b$, $c$, and $d$ such that the sum of the square of the amplitudes is one. (More on this below.)

We may also write complex numbers as $r \times e^{i \theta}$, utilizing the formula $e^{ix} = \cos x + i \times \sin x$ and the fact that $a + i \times b$ is giving us coordinates for a point in a plane. This form is sometimes more convenient, especially because if $\alpha = r \times e^{i\theta}$, then

$$
\left| \alpha \right|^2 = \left(r \times e^{i\theta}\right)\left(r \times e^{-i\theta}\right) = r^2 \times e^{i\theta - i\theta} = r^2
$$

Superposition is really weird, but it isn’t obviously _useful_ for one spin. It turns out to become *very* useful for multiple spins because now the "state space" we have to work with is much larger.

Having multiple spins to work with leads us naturally to *entanglement.*

### Entanglement

To combine quantum systems we use *tensor products*. For two spins, instead of just `U` and `D`, we have four basis states: `UU`, `UD`, `DU`, and `DD`.

$$
|U\rangle \otimes |U\rangle = |UU\rangle
$$

A two-spin system can be in a superposition of all four. Three spins have $2^3 = 8$ basis states, four have $2^4 = 16$, and in general N spins have $2^N$. The state space grows *exponentially* with the number of qubits.

We’ll talk about how to *create* entanglement later, but first let’s just think about how many parameters we need to describe a state. For a single spin, it turns out we need two. Why?

We have two complex parameters, say alpha and beta - this is four real parameters. But, we have a normalization constraint and an overall phase that cannot be observed. This means if we have:

$$
|\Psi\rangle = r_1 e^{i\theta_1} |U\rangle + r_2 e^{i\theta_2} |D\rangle
$$

we may "pull out" an overall phase that will **always** disappear during a measurement.

$$
|\Psi\rangle = e^{i\theta_1} \left(r_1 |U\rangle + r_2 e^{i\theta_2 - i\theta_1} |D\rangle \right) → r_1 |U\rangle + r_2 e^{i\Delta \theta} |D\rangle
$$

So it is only the *relative* phase that is meaningful. Recall that normalization means:

$$
r_1 = \sqrt{1 - r_2^2} .
$$

So, four parameters minus two constraints leaves us with two free parameters.
What about for two spins? We write

$$
|\Psi\rangle = \alpha|UU\rangle + \beta|UD\rangle + \gamma|DU\rangle + \kappa|DD\rangle
$$

and start with real 8 parameters. How many constraints are there?

We again have a normalization condition and an overall phase factor we can pull out --- two constraints.

This means we need 6 parameters. Note - 6, not 4! We might have expected that if we need 2 parameters for one qubit that 2x2=4 parameters would have described two qubits, but there are two extra parameters! What is going on? What do the two extra terms describe? They describe *entanglement* (or at least the possibility of entanglement).

<img src="images/universal-eye-4bhhwmsYl-c-unsplash.jpg" height=500>

Photo by [Universal Eye](https://unsplash.com/@universaleye)

_Physically entanglement is the combination of two systems such that they are no longer independent systems_. New degrees of freedom arise as a consequence. Consider the following state

$$
|s\rangle = \frac{1}{\sqrt{2}} \left(|UD\rangle - |DU\rangle \right).
$$

This is a state in which we have no information about either spin - they are both equally likely to be up or down and we can't say anything until we measure them. But, we have *complete knowledge about the pair* - we know they are opposite!

There is no analog of entanglement in classical mechanics. How can we have a system where we have complete knowledge of the aggregate, but now nothing about the parts? It would be like saying, “I know everything about that car over there - how fast it is going, which doors are open, whether there is music playing over its speakers, etc., but I know nothing about the individual parts that make up the car.”

### The Power of Quantum

**Superposition and entanglement create the power of quantum computing.** Many people mistakenly believe that quantum computers are somehow running multiple calculations in parallel and it is massive parallelism that supplies the speed ups. Even good textbooks sometimes use this description, and it is a somewhat useful way of talking in "short hand" about quantum computing to people that aren't familiar with superposition and entanglement. But this description isn’t *correct*, and wouldn’t lead to exponential speed ups either thanks to [Amdahl's Law](https://en.wikipedia.org/wiki/Amdahl%27s_law).

What quantum computers *do* offer is the possibility of creating very entangled states with a **linear** number of operations, and these states live in an **exponentially large** space. Then, we can manipulate the **entire distribution** of amplitudes with **single operations** on single qubits.

Measurements are samples of the probability distributions encoded by the amplitudes, which means computations are not deterministic. That means we don't get the same answer every time we run the program! Instead, we need to run a calculation many times to estimate all the amplitudes, and at best we can only asymptotically approach the true values.

So **what we are really doing with a quantum computer is very efficiently creating a probability distribution that represents our problem, and very efficiently sloshing the entire distribution around during a calculation.** Then we **sample from the distribution** we calculated. The **goal of the calculation is to try to raise the probability of getting the right answer.**

This is discussed beautifully in this [Saturday Morning Breakfast Cereal comic](https://www.smbc-comics.com/comic/the-talk-3). I'll also point out another cartoon I like from them - [be careful with hype!](https://www.smbc-comics.com/comic/quantum)

## Computation is performed by physical systems

Okay, now we're ready for quantum computing!

Let's start with the big picture... What is _computation_?

<img src="images/matrix-code-rain1.jpg" height=500>

**Computation is manipulating a physical system according to rules and interpreting the result.** Can you think of any computation that doesn't involve a physical system?

An abacus is a good example. Move the beads following certain rules, and the new positions represent an arithmetic result. The abacus doesn't "know" it added numbers — but we do.

<img src="images/giphy.gif">

But an abacus is slow. Well - some people are [really fast](https://www.youtube.com/watch?v=GQtqlB-jXO0) with them. But, today we use electronic devices.

To add two numbers on an electronic device we first develop an _algorithm_, which tells us how to relate an arithmetic operation to the manipulation of binary numbers (two-level systems!) and then we _simulate_ that algorithm by running currents and measuring voltages across physical transistors. Finally, we _interpret_ the outcome (using our brain, starting a [turtles all the way down chain](https://en.wikipedia.org/wiki/Turtles_all_the_way_down)...).

Note that there is some philosophy here! You may be wondering "why are we talking about quantum _computing_ in a physics class?" - but if computation _is_ the simulation of algorithms by _physical_ systems...

<img src="images/FaiPR_LXEAAU5uz.jpeg">

From a certain point of view, physics _is_ computing and computing _is_ physics. There is a school of thought that at the bottom of everything we will find [information theory](https://en.wikipedia.org/wiki/Information_theory) — and it will necessarily be _quantum_ information theory.

A quantum computer is just an instrument that plays the base layer of physical reality — and that sounds like something that belongs in a physics class!

## Classical circuit model

There are many universal models for computation (Turing machines, circuits, and so on.) Let's look at the circuit model because it turns out to be the way people reason about quantum programs today.

### Start with: Addition with binary numbers

<img src="images/10kindsofpeople.jpg">

To add two numbers with a computer (like a smartphone or a laptop), it first represents them in [_binary_](https://en.wikipedia.org/wiki/Binary_number). Binary numbers are base-2. Most humans are used to base-10, but base-2 (and base-16) are very useful in computer science. Base 10 looks like this:

```
765 = 7 x 10^2 + 6 x 10^1 + 5 * 10^0
```

Note that base-10 is so ingrained in the way we think that we use base-10 to explain base-10!

Binary works the same way, but with powers of two instead of ten:

```
0b101 = 1 x 2^2 + 0 x 2^1 + 1 x 2^0 = 5
```

Digits range between 0 and 1 instead of 0 and 9. Each place value is a power of two, read right to left.

Binary is useful for computation with physical systems because we can represent binary numbers with two-level systems, and two-level systems are _physically_ simple. There are many _electronic_ two-level systems that we can manipulate very quickly.

For addition --- first, we "hardwire" some simple rules:

<img src="images/add_two_binary_numbers_rules.png" height="90">

<img src="images/add_three_binary_numbers_rules.png" height="90">

To add two binary numbers, we store them in registers — physical two-level systems like voltage levels or tiny magnets. A third register holds the result, plus a carry bit. We scan across the digits, applying our addition rules:

Here we add 6 and 7:

<img src="images/binary_addition.png" height="150">

For more, see this [web app](https://chortle.ccsu.edu/AssemblyTutorial/zAppendixE/binaryAdd.html).

In [ ]:
0b110 + 0b111

In [ ]:
bin(13)

So really, there are 0b10 kinds of people in the world.

Now, let's build some _circuits_...

### Logic gates

Now that we have these rules for binary addition, how do we _implement_ them in hardware? Humans have figured out how to use [transistors to implement _logic gates_](https://en.wikipedia.org/wiki/Transistor–transistor_logic). We represent these gates visually as circuits - lines carry data (classical binary digits - 0's and 1's) and the gates themselves are symbols with line inputs and outputs. We use _truth tables_ to show the action of the gate on the data in the line.

<img src="images/LogicCircuitsWithTables1.png" alt="Logic circuits">

Some composite gates are so popular they are often built as primitives:

<img src="images/LogicCircuitsWithTables2.png" alt="Logic circuits">

You can see how you would implement a table like this one with logic gates:

<img src="images/add_two_binary_numbers_rules.png" height="90">

You need two inputs and two outputs. This function (made of an AND and an XOR) is called a _Half Adder_:

<img src="images/HalfAdderWithTable.png" alt="Half Adder">

We may combine Half Adders and an OR gate to build a _Full Adder_:

<img src="images/FullAdderWithTable.png" alt="Full Adder">

We may cascade Full Adders to add even larger numbers. So, for example, for two 3-bit numbers, we have:

<img src="images/CascadedAdder.png" alt="Cascaded Adders">

**Exercise:** Verify this circuit performs addition correctly.

**Exercise:** Can you work out a circuit for binary multiplication?

We're near the bottom of the software stack — logic gates are built directly into hardware. When you hear a chip has billions of transistors, that means billions of these gates and combinations, churning out answers as electrical current flows through them.

<img src="images/m1s.jpeg" alt="M1 chip family" height=400>

From there, you layer on abstractions — bundling circuits for complex tasks, hardcoding common operations into silicon, and keeping others flexible. Amazingly, the bottom layer can be very simple: you can do everything using **only** `NAND` gates!

At the end you have the power to write expressive programs that can, in principle, compute anything!

## Limits to "classical" computing

Can we really compute _everything_ under this paradigm? In principle, yes — but there are practical limits. We have to _build a physical system_ that simulates an algorithm, and if that requires too many steps, the calculation may take longer than the lifetime of the universe.

For example, we might know the algorithm to find a molecule that cures cancer, but if simulating it is physically infeasible, we can never learn what that molecule actually is!

Suppose we want perform a simulation of a molecule to compute chemical properties (e.g. some protein binding affinity). Electrons are two-state _quantum_ systems! This means that in order to write down the physical description of a set of electrons (which features _superposition_ and _entanglement_), we need to do a lot of book-keeping. In fact, the set of numbers we need to write down the state of the system scales like $2^N$, where $N$ is the number of electrons.

Time to wave my hands... (numbers are round and rough...)

<img src="images/giphy_5.gif" alt="Jazz hands">

Imagine we have 50 electrons in our molecule (that matter for its chemistry). If one number were one byte (8) bits of precision, 50 electrons requires many Gigabytes of memory:

In [ ]:
2**50 / 8 / 1e9

This is roughly the scale of a world-class supercomputer, so humans **can** do this. Suppose you come up with a new idea for a molecule with 100 important electrons. If $2^{50} \simeq 1$ world-class supercomputer, how many world-class supercomputers is $2^{100}$ equal to?

In [ ]:
print("{:1.5g}".format(2**100 / 2**50))

That is a million billion supercomputers! Human civilization has, maybe, three supercomputers of this class, so... a million billion of them (networked with shared memory) seems far off. And what if we needed even more electrons? **Exponential functions get REALLY big, REALLY fast!**

Conventional transistor-based hardware must brute-force the bookkeeping of superposition and entanglement, and the cost scales **exponentially** with system size. Can [Moore's Law](https://en.wikipedia.org/wiki/Moore%27s_law) save us? Computing power has roughly doubled every two years for decades — can we just wait?

<img src="images/giphy_4.gif">

Moore's Law has stubbornly held for decades, but transistors are approaching single-atom sizes — there isn't much room left. Worse, we hit a more immediate wall: [Dennard scaling](https://en.wikipedia.org/wiki/Dennard_scaling) has already broken down. We can't _turn on_ the whole chip or it will melt.

<img src="images/limits_to_classical_comp_m_alam_rigetti.png" height=400>

Figure courtesy of M. Sohaib Alam.

The resources we'd need are ludicrously beyond civilization's reach. But we'd really like to solve these problems — cancer sucks!

Maybe the problem is the hardware. Instead of brute-forcing quantum mechanics with classical logic gates, what if we built a computation device out of _quantum systems themselves_ — letting nature do the heavy lifting?

Good idea! Doing this sort of experiment with the molecules is called "chemistry", and it is _great_. The problem is these "simulators" are not generically programmable. We can set up experiments to answer a _specific question_, but we can't automate them or put them into other algorithms as subroutines. And we need the stuff too, the _specific_ atoms and molecules. What if they are rare, hard to get, or very expensive to synthesize?

What we want is an extension to our current paradigm that gives us more simulation power or one that lets us get around some of the physical limitations we're dealing with.

## Distinguishing features of Quantum computing

As we saw, simulating quantum systems classically is hard because superposition and entanglement demand exponentially many numbers to track. The physics of our simulator are just wrong for the problem.

What if the computing device itself were quantum? Our best candidate: a _quantum computer_.

Quantum computing manipulates _quantum states_ instead of voltage levels or wooden beads. The key difference: we use **qubits** instead of classical bits. Recall that qubits access _superpositions_ and can be _entangled_, so the state space for $n$ qubits is $2^n$. This exponential scaling is exactly what we need.

<img src="images/qubit_scaling_m_alam_rigetti.png">

Figure courtesy of M. Sohaib Alam.

The physical manifestation of our machine is very different than what we're used to. For example, our "computer" might involve a set of spins (in a "register") that we manipulate (e.g. trapped ions with laser pulses), moving from one quantum state to another.

<img src="images/registerofqubits.png">

Here are some photos of real quantum computing devices:

<img src="images/pictures-of-quantum-hardware.png">

From left to right: Google's superconducting transmon chip ([Sycamore](https://ai.googleblog.com/2019/10/quantum-supremacy-using-programmable.html)), Fermilab's [superconducting radio frequency cavity](https://sqms.fnal.gov/research/), Quantinuum's [ion trap](https://www.quantinuum.com/hardware), Xanadu's [photonic chip](https://www.xanadu.ai/hardware).

Generally we need to cool quantum devices down to very close to absolute zero (although not always - e.g., ion traps can operate at room temperature). This requires the use of [dilution refrigerators](https://en.wikipedia.org/wiki/Dilution_refrigerator), which look cool and show up in press photos all the time.

Here are a couple of "dil fridges" - on the left Fermilab scientists Dan Bowring and Rakshya Khatiwada are working on a fridge that will use qubits as _detectors_ in physics experiments, and on the right we have a "promotional" photo from [Bluefors](https://bluefors.com/):

<img src="images/dil-fridges.png">

### What are quantum computers practically good for?

It is difficult to say what the real "killer app" is at this point. Microsoft has put together a [slick video series](https://www.youtube.com/playlist?list=PLFPUGjQjckXFsOEBvvaDeIk5GxctP0ZhX) you can watch sometime for one (very corporate, very buzz-word laden) take on the question.

One thing we can be sure they will be good for though is _simulating quantum systems_, e.g.:

* Chemicals (e.g. drugs),
* Materials,
* Black holes (?),
* Nuclear matter,

and so on. Why would we need a quantum computer to simulate these systems? Fundamentally quantum systems must be described using quantum mechanics, which means we must allow for superposition and entanglement, and we must be able to compute with these objects.

**From this perspective a quantum computer is first and foremost a _tool for discovery_, like a telescope or a particle accelerator.**

(My take on this meme on Threads...)

<img src="images/yelling_meme2.png" height="675" alt="meme-joke">

### Qubits

Recall that qubits can be superpositions of 0 and 1. Now let's formalize this with a **state vector**:

$$
|\psi\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix} = \alpha |0\rangle + \beta |1\rangle
$$

Here $|0\rangle$ and $|1\rangle$ are _basis states_ in the **computational basis** — think of them as "heads" and "tails."

We call $|0\rangle$ the "zero qubit" and $|1\rangle$ the "one qubit." As before, $\alpha$ and $\beta$ are complex numbers obeying normalization:

$$
\alpha^* \alpha + \beta^* \beta = 1
$$

where $\alpha^*$ is the _complex conjugate_ (flip the sign of the imaginary part). This guarantees each product is real.

### The Bloch Sphere

We often use a construct called the [Bloch Sphere](https://en.wikipedia.org/wiki/Bloch_sphere) to represent a (single) qubit visually. The Bloch sphere gives us a way to visualize how qubits with different $\alpha$ and $\beta$ values "look" by mapping those points onto a two-dimensional surface (best visualized as a point on a sphere):

In [ ]:
plot_bloch(zero_qubit)

In [ ]:
plot_bloch(one_qubit)

Suppose we choose an arbitrary $\alpha$ and $\beta$ pair? Recall the restriction that $\alpha^2 + \beta^2$ must equal 1.

In [ ]:
eighty_twenty_qubit = np.sqrt(0.8)*zero_qubit - i_ * np.sqrt(0.2)*one_qubit
plot_bloch(eighty_twenty_qubit)

**Exercise:** Play with this code and draw some new qubits. Recall that:

$$
\alpha^2 + \beta^2 = 1
$$

really means

$$
\alpha^* \alpha + \beta^* \beta = 1
$$

where $\alpha^*$ is the _complex conjugate_ of $\alpha$. Here, that just means $i \to -i$, so

$$
(2 + 3i)^* = 2 - 3i
$$

etc.

### Measurement

Recall that measurement collapses a superposition. Formally: we choose a _basis_ (e.g., the computational basis $|0\rangle$, $|1\rangle$), and the measurement always returns a state in that basis. The probability of each outcome is the squared amplitude. After collapse, repeated measurements give the same result.

For example, if $|\psi\rangle = \sqrt{0.8}|0\rangle + \sqrt{0.2}|1\rangle$, we have an 80% chance of measuring $|0\rangle$ and a 20% chance of measuring $|1\rangle$. After measurement, the superposition is gone.

Quantum mechanics is weird! Read about how this was discovered with the [Stern–Gerlach experiment](https://en.wikipedia.org/wiki/Stern%E2%80%93Gerlach_experiment).

## Quantum circuits - lines and gates

### Special rules

Quantum circuits have data lines and gates like classical circuits, but with two key constraints:

* **No copying** — we can't duplicate a qubit (we'll see why later).
* **Reversibility** — all gates are [unitary](https://en.wikipedia.org/wiki/Unitary_operator), so every circuit can be run backwards. This rules out gates like classical `AND`. It also means gates are _norm preserving_.

Here is a quantum circuit:

<img src="images/epr-circuit.png" height="350">

We'll build this circuit in code later and explain the symbols. The `H` and the cross-bar both represent _quantum gates_ (the analog of our classical logic gates above). The `H` is acting on one qubit while the cross-bar is acting on two. This circuit is actually creating a _Bell pair_.

### Gates are linear operators

Linear algebra is the foundation of quantum computing. (It is the foundation of a _lot_ of things.)

We can represent quantum gates with _matrices_. Acting with a gate is equivalent to multiplying a state vector with a matrix.

For single qubits, these gates are $2 \times 2$ _Unitary matrices_.

First, we need to know how to take the _dot product_ of two vectors:

$$\begin{split}
A =& \begin{pmatrix} a_1 \\ a_2 \end{pmatrix};~ B = \begin{pmatrix} b_1 \\ b_2 \end{pmatrix} \\
A \cdot B =&~ a_1 \times b_1 + a_2 \times b_2
\end{split}$$

The dot product is used to define the _norm_, or length, of a vector:

$$\rightarrow |A| = \sqrt{A \cdot A} = \sqrt{a_{1}^2 + a_{2}^2}$$

This should actually look familiar...

<img src="images/pythag.png" height=400>

We often express inner products using the "bra-ket" ("bracket") notation:

$$
A \cdot B = \langle A | B \rangle
$$

$| 0 \rangle$ and $| 1 \rangle$ are _orthonormal_, which means:

$$
\begin{split}
\langle 0 | 0 \rangle =&~ \langle 1 | 1 \rangle = 1 \\
\langle 0 | 1 \rangle =&~ \langle 1 | 0 \rangle = 0
\end{split}
$$

And therefore, for $|\psi\rangle = \alpha |0\rangle + \beta |1\rangle$:


$$
\begin{split}
\langle \psi | \psi \rangle =&~ \left(\alpha \langle 0| + \beta \langle 1|\right) \left(\alpha |0\rangle + \beta |1\rangle \right) \\
=&~ \alpha^2 \langle 0 | 0 \rangle + \alpha \beta \left( \langle 0 | 1 \rangle + \langle 1 | 0 \rangle \right) + \beta^2 \langle 1 | 1 \rangle \\
=&~ \alpha^2 + \beta^2
\end{split}
$$

Since we know $\langle \psi | \psi \rangle = 1$, this is another way of expressing the normalization condition $\alpha^2 + \beta^2 = 1$.

Next, matrix multiplication. We dot each row of the first matrix with the column of the second. For 2×2 matrices acting on 2×1 vectors:

$$
\begin{pmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{pmatrix} \begin{pmatrix} b_{1} \\ b_{2} \end{pmatrix} = \begin{pmatrix} a_{11} b_1 + a_{12} b_2 \\ a_{21} b_1 + a_{22} b_2 \end{pmatrix}
$$

The output is a 2×1 vector — a gate acting on a state gives a new state. Note that the order of multiplication matters.

**Exercise:** Multiply

$$
\begin{pmatrix} 3 & 2 \\ 4 & -1 \end{pmatrix} \begin{pmatrix} 3 \\ 5 \end{pmatrix}
$$

### Multiple qubits

Multi-qubit states are also state vectors. A two-qubit system can be in a superposition of all $2^2 = 4$ basis states:

$$
\psi = a|00\rangle + b|01\rangle + c|10\rangle + d|11\rangle
$$

Four complex amplitudes is manageable, but a molecule might need hundreds of qubits — and the state space grows as $2^n$:

In [ ]:
pretty_print_number(2**200)

Of course, there is still a normalization condition on these terms. For example:

$$
\psi = \frac{1}{\sqrt{2}} \left( |001\rangle + |110\rangle \right)
$$

The sum of the squares of the coefficients is 1. This is _always_ the case for a valid quantum state.

Note that we cannot say the leftmost qubit is definitely 0 or 1 — it is genuinely in a combination of both. This isn't just ignorance; experiments rule out [hidden variables](https://en.wikipedia.org/wiki/Bell_test). A quantum state really can be a combination of states.

### Tensor products

How do we actually write the state vector for multi-qubit states? We need to take the _tensor product_ of the qubits, e.g.

$$
|\alpha\rangle |\beta\rangle \equiv |\alpha\beta\rangle \equiv |\alpha\rangle \otimes |\beta\rangle .
$$

_This is not the same as a dot or cross-product. It has different rules!_ In particular, we need a multiplication operation that takes two two-element vectors and makes a four-element vector. Three qubits should combine to make an eight-element vector.

The clearest way to show how the tensor product works is with an example:

$$
\begin{pmatrix} a \\ b \end{pmatrix} \otimes \begin{pmatrix} c \\ d \end{pmatrix} =  
\begin{pmatrix} a \begin{pmatrix} c \\ d \end{pmatrix} \\ b \begin{pmatrix}  c \\ d \end{pmatrix} \end{pmatrix} =
\begin{pmatrix} ac \\ ad \\ bc \\ bd \end{pmatrix}
$$

So,

$$
|0\rangle |0\rangle =
\begin{pmatrix} 1 \\ 0 \end{pmatrix} \otimes \begin{pmatrix} 1 \\ 0 \end{pmatrix} =  
\begin{pmatrix} 1 \begin{pmatrix} 1 \\ 0 \end{pmatrix} \\ 0 \begin{pmatrix}  1 \\ 0 \end{pmatrix} \end{pmatrix} =
\begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix} =
|00\rangle
$$

This generalizes to higher dimensions in the "obvious" way, e.g.

$$
\begin{pmatrix} a_1 & b_1 \\ c_1 & d_1 \end{pmatrix} \otimes \begin{pmatrix} a_2 & b_2 \\ c_2 & d_2 \end{pmatrix} =  
\begin{pmatrix} a_1 \begin{pmatrix} a_2 & b_2 \\ c_2 & d_2 \end{pmatrix} & b_1 \begin{pmatrix} a_2 & b_2 \\ c_2 & d_2 \end{pmatrix} \\ c_1 \begin{pmatrix} a_2 & b_2 \\ c_2 & d_2 \end{pmatrix} & d_1 \begin{pmatrix} a_2 & b_2 \\ c_2 & d_2 \end{pmatrix} \end{pmatrix} =
\begin{pmatrix}
a_1 a_2 & a_1 b_2 & b_1 a_2 & b_1 b_2 \\
a_1 c_2 & a_1 d_2 & b_1 c_2 & b_1 d_2 \\
c_1 a_2 & c_1 b_2 & d_1 a_2 & d_1 b_2 \\
c_1 c_2 & c_1 d_2 & d_1 c_2 & d_1 d_2 \\
\end{pmatrix}
$$

**Exercise:** work out $|0\rangle \otimes |1\rangle$ and $|1\rangle \otimes |1\rangle$

<img src="images/giphy_2.gif">

### Quantum Gates

One example of a valid quantum gate is the _Not_ gate, also called the _X_ gate:

$$
X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}
$$

Multiple gate operations in sequence are equivalent to multiplying the matrices in order (the order matters).

If we write it out, we see the Not gate flips one computational basis state to the other:

$$
X |0\rangle = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 0 \\ 1 \end{pmatrix} = |1\rangle
$$

Note that this implies $XX = I$. Two applications of $X$ bring us back to the identity.

$$
\begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}
$$

Another example of a valid quantum gate is the _Y_ gate:

$$
Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}
$$

Note that $XY$ is not the same as $YX$:

$$
XY = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} = \begin{pmatrix} i & 0 \\ 0 & -i \end{pmatrix}
$$

$$
YX = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} -i & 0 \\ 0 & i \end{pmatrix}
$$

Another important gate is the _Hadamard_. It rotates basis states into superpositions:

$$
H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}
$$

$$
H \psi = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} \times \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ 1 \end{pmatrix}
$$

$$
\begin{split}
H |0\rangle =&~ \frac{1}{\sqrt{2}} \left(|0\rangle + |1\rangle \right) \equiv |+\rangle  \\
H |1\rangle =&~ \frac{1}{\sqrt{2}} \left(|0\rangle - |1\rangle \right) \equiv |-\rangle  \\
\end{split}
$$

Note that like $X$, $H$ is also its own inverse, such that $HH = I$.

**Exercise:** Work out $HH$ to prove $HH = I$.

In [ ]:
plot_bloch(X * zero_qubit)

In [ ]:
plot_bloch(H * zero_qubit)

In [ ]:
plot_bloch(H * H * zero_qubit)

### Two-qubit gates

The gate for a two qubit state is a 4x4 matrix. One important two qubit gate is the CNOT, or _Controlled Not_ gate. This gate applies a Not to the second qubit in the pair, but only if the first qubit is in the $|1\rangle$ state. The CNOT gate looks like:

$$
\begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0 \\
\end{pmatrix}
$$

Presented like this, the matrix doesn't make much sense, but look at how two-qubit state vectors are written:

$$
|00\rangle =
\begin{pmatrix}
1 \\
0 \\
0 \\
0 \\
\end{pmatrix},
|01\rangle =
\begin{pmatrix}
0 \\
1 \\
0 \\
0 \\
\end{pmatrix},
|10\rangle =
\begin{pmatrix}
0 \\
0 \\
1 \\
0 \\
\end{pmatrix},
|11\rangle =
\begin{pmatrix}
0 \\
0 \\
0 \\
1 \\
\end{pmatrix},
$$

If you stare at that long enough and think back to the rules we presented for matrix multiplication, you should be able to convince yourself that the CNOT only changes the state of vectors where the first qubit is in state $|1\rangle$ and its action is to flip the state of the second qubit.

The CNOT gate is important because it lets us generate _entanglement_.

### Bell pairs and entanglement

_Entanglement_ occurs when a multi-qubit state _cannot be decomposed into a product of single-qubit states_.

Many multi-qubit quantum states _may_ be written as the decomposition of single qubit states. For example, $|00\rangle$:

$$
\begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix} = |00\rangle = |0\rangle \otimes |0\rangle
$$

This is just a fancy way of saying the state $|00\rangle$ is made up of two individual qubits in the $|0\rangle$ state sitting next to each other.

All classical states may be written as decompositions in this way, but there are quantum states that cannot be decomposed. We call these states _entangled_. For example, the state

$$
\frac{1}{\sqrt{2}} \left( |00\rangle + |11\rangle \right) =
\frac{1}{\sqrt{2}} \left(
    \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix} +
    \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}
    \right) =
\frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ 0 \\ 0 \\ 1 \end{pmatrix}
$$

cannot be broken down into a tensor product of single qubit states. We happen to call two qubits in this particular state a "Bell pair". (There are actually four different Bell pairs and they form a _basis_.)

**Exercise:** Try to find a tensor product of two single qubit states that produces a Bell pair. You won't find one! See if you can prove it _isn't possible_ to produce the state that way.

**Exercise:** Can you find the three other Bell basis states? Hint - we get the first one by operating on $|00\rangle$ with the Bell circuit.

### No cloning!

<img src="images/giphy_6.gif">

(We don't mean that kind of clone...)

One last thing before we start writing quantum circuits. A very important property of qubits is - we can't copy them! (In general.) This is a consequence of something called the [no cloning theorem](https://en.wikipedia.org/wiki/No-cloning_theorem).

Of course, if you know your qubit is in one of the basis states (after a measurement for example), you can easily "copy" it by putting another qubit into that state, but you can't copy an arbitrary qubit.

Suppose we *did* have a unitary operator that could clone a state. This means we have some operator $U$ such that $U|\psi\rangle |0\rangle = |\psi\rangle |\psi\rangle$. This can only be true if $|\psi\rangle$ is $|0\rangle$ or $|1\rangle$. Everything hinges on the fact our operator is _unitary_ (which means it can't change the norm of our state).

Here's a quick proof:

$$
\begin{split}
\langle \phi | \psi \rangle \langle 0 | 0 \rangle =&~ \langle \phi | \langle 0| | \psi \rangle | 0 \rangle \\
=&~ \langle \phi | \langle 0| U^{\dagger} U | \psi \rangle | 0 \rangle \\
=&~ \langle \phi | \psi \rangle \langle \phi | \psi \rangle \\
=&~ \left( \langle \phi | \psi \rangle \right)^2
\end{split}
$$

This is only true if $\langle \phi | \psi \rangle$ is 0 or 1.

## Quantum Error Correction

### Why error correction?

We noted earlier that real quantum hardware is noisy — gates are imperfect and qubits interact with their environment. Classical computers face noise too, but they solve it with redundancy: store each bit multiple times and take a majority vote. This works because classical bits can be freely copied.

For a thorough introduction to both classical and quantum error correction, see Girvin's [lecture notes](https://arxiv.org/abs/2111.08894).

### Why is quantum error correction hard?

Three features of quantum mechanics make error correction fundamentally harder than in the classical case:

1. **No cloning** — we just proved we can't copy an arbitrary qubit, so naive redundancy is out.
2. **Measurement destroys superpositions** — if we measure a qubit to check for errors, we collapse it and lose the computation.
3. **Errors are continuous** — a classical bit flips from 0 to 1, but a qubit can rotate by any small angle. We need to correct a continuum of possible errors.

The key insight of quantum error correction is that we can detect errors _indirectly_ by measuring **correlations** between qubits (called _stabilizers_ or _parity checks_) without ever learning the encoded state itself.

### The repetition code

The simplest error-correcting code — classical or quantum — is the **repetition code**. We encode one logical bit using multiple physical bits. If noise flips one of them, a majority vote recovers the original.

The quantum version protects against **bit-flip errors** (X errors). We encode one logical qubit in three physical qubits:

$$|0_L\rangle = |000\rangle, \quad |1_L\rangle = |111\rangle$$

To detect errors without measuring the data qubits directly, we measure **parity checks** between neighboring pairs. These tell us whether adjacent qubits agree or disagree — without revealing what value they hold.

The parity checks (stabilizers) for the 3-qubit repetition code are:

$$Z_0 Z_1 \quad \text{and} \quad Z_1 Z_2$$

Each measures whether a pair of qubits is in the same state (+1) or different states (−1). The pair of measurement outcomes is called the **syndrome**. From the syndrome, we can deduce which qubit (if any) was flipped:

| Syndrome ($Z_0Z_1$, $Z_1Z_2$) | Interpretation |
|:---:|:---|
| +1, +1 | No error |
| −1, +1 | Error on qubit 0 |
| −1, −1 | Error on qubit 1 |
| +1, −1 | Error on qubit 2 |

The notation $Z_0 Z_1$ is shorthand — since we have three qubits, the full operator is $Z_0 \otimes Z_1 \otimes I_2$ (identity on the third qubit). Recall that $Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$. Using the tensor product rules from earlier:

$$
Z_0 \otimes Z_1 \otimes I_2 =
\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \otimes
\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix} \otimes
\begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} =
\begin{pmatrix}
1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & -1 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & -1 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & -1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & -1 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 1
\end{pmatrix}
$$

This is a diagonal matrix with $+1$ on entries where qubits 0 and 1 agree (both $|0\rangle$ or both $|1\rangle$) and $-1$ where they disagree. That's exactly the parity check we need — and the identity on qubit 2 means we learn nothing about it.

To see this in action, apply the operator to $|000\rangle$ (qubits 0 and 1 agree):

$$
(Z_0 \otimes Z_1 \otimes I_2)|000\rangle =
\begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \end{pmatrix}
= +1 |000\rangle
$$

Now apply it to $|010\rangle$ (qubits 0 and 1 disagree):

$$
(Z_0 \otimes Z_1 \otimes I_2)|010\rangle =
\begin{pmatrix} 0 \\ 0 \\ -1 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \end{pmatrix}
= -1 |010\rangle
$$

The $+1$ or $-1$ eigenvalue tells us the parity without revealing the actual qubit values — exactly what we need for error detection.

Before we simulate, let's look at the circuit for measuring the two stabilizers. We use [Cirq](https://quantumai.google/cirq) to draw it:

In [ ]:
# Repetition code stabilizer measurement circuit
q0, q1, q2 = cirq.LineQubit.range(3)          # data qubits
a0, a1 = cirq.NamedQubit('s01'), cirq.NamedQubit('s12')  # syndrome ancillas

rep_code_circuit = cirq.Circuit([
    # Stabilizer Z0*Z1: parity of q0 and q1 into ancilla s01
    cirq.Moment([cirq.CNOT(q0, a0), cirq.CNOT(q1, a1)]),
    cirq.Moment([cirq.CNOT(q1, a0), cirq.CNOT(q2, a1)]),
    # Measure ancillas
    cirq.Moment([cirq.measure(a0, key='Z0Z1'),
                 cirq.measure(a1, key='Z1Z2')]),
])

print(rep_code_circuit)

Let's simulate this with [Stim](https://github.com/quantumlib/Stim), a fast stabilizer circuit simulator designed for quantum error correction:

In [ ]:
# 3-qubit repetition code with Stim
repetition_code = stim.Circuit('''
    # Data qubits: 0, 1, 2
    # Ancilla qubits: 3, 4

    # Introduce a single bit-flip error on qubit 1
    X_ERROR(1.0) 1

    # Stabilizer Z0*Z1: CNOT from data qubits 0,1 into ancilla 3
    CX 0 3
    CX 1 3
    # Stabilizer Z1*Z2: CNOT from data qubits 1,2 into ancilla 4
    CX 1 4
    CX 2 4

    # Measure ancillas to get the syndrome
    M 3 4
''')

sampler = repetition_code.compile_sampler()
syndrome = sampler.sample(shots=1)
print(f'Syndrome: {syndrome[0].astype(int)}')
print()
s0, s1 = syndrome[0][0], syndrome[0][1]
if s0 and s1:
    print('Both stabilizers triggered -> error on qubit 1 (middle)')
elif s0:
    print('First stabilizer triggered -> error on qubit 0')
elif s1:
    print('Second stabilizer triggered -> error on qubit 2')
else:
    print('No stabilizers triggered -> no error detected')

Now let's make it realistic — instead of placing an error deterministically, we'll use a random noise model and run many shots to see the statistics:

In [ ]:
# Repetition code with random noise
noisy_repetition_code = stim.Circuit('''
    # Apply random bit-flip errors with 10% probability
    X_ERROR(0.1) 0 1 2

    # Measure stabilizers
    CX 0 3
    CX 1 3
    CX 1 4
    CX 2 4
    M 3 4
''')

sampler = noisy_repetition_code.compile_sampler()
syndromes = sampler.sample(shots=10000)

from collections import Counter
counts = Counter(tuple(row.astype(int)) for row in syndromes)
for syndrome, count in sorted(counts.items()):
    pct = count / 100
    label = {
        (0, 0): 'no error',
        (1, 0): 'error on q0',
        (1, 1): 'error on q1',
        (0, 1): 'error on q2',
    }.get(syndrome, 'multi-qubit error')
    print(f'  Syndrome {syndrome}: {count:5d} ({pct:5.1f}%) - {label}')

With a 10% error rate per qubit, most of the time (~73%) no error occurs. Single-qubit errors are caught by the syndrome. The rare case where two or more qubits flip simultaneously can lead to a _logical error_ — the syndrome points to the wrong qubit. This is why we need the physical error rate to be well below 50%: the code only helps if multi-qubit errors are rare.

### The surface code

The repetition code only corrects bit-flip (X) errors. Real quantum errors also include **phase flips** (Z errors) and combinations of both. The **surface code** is the leading architecture for fault-tolerant quantum computing because it corrects both types of errors simultaneously.

The surface code arranges data qubits on a 2D grid with two types of **stabilizer checks**:

- **X stabilizers** (red) detect Z errors
- **Z stabilizers** (blue) detect X errors

Each stabilizer measures the parity of its neighboring data qubits — the same idea as the repetition code, but now in two dimensions and for two error types. Let's use Stim to generate a distance-5 rotated surface code and visualize its layout:

In [ ]:
import re
import matplotlib.patches as mpatches

# Generate a distance-5 rotated surface code
surface_circuit = stim.Circuit.generated(
    'surface_code:rotated_memory_z',
    distance=5,
    rounds=1,
)

# Extract qubit coordinates
all_qubits = {}
for line in str(surface_circuit).split('\n'):
    m = re.match(r'QUBIT_COORDS\((\d+),\s*(\d+)\)\s+(\d+)', line)
    if m:
        x, y, idx = int(m.group(1)), int(m.group(2)), int(m.group(3))
        all_qubits[idx] = (x, y)

# Data qubits get M; ancilla qubits get MR
data_idx, measure_idx = set(), set()
for line in str(surface_circuit).split('\n'):
    if line.startswith('MR '):
        measure_idx.update(int(s) for s in line[3:].split())
    elif line.startswith('M '):
        data_idx.update(int(s) for s in line[2:].split())

# X ancillas are those that receive H gates
x_anc = set()
for line in str(surface_circuit).split('\n'):
    if line.startswith('H '):
        x_anc.update(int(s) for s in line[2:].split())

# Plot
fig, ax = plt.subplots(1, 1, figsize=(7, 7))
for idx in measure_idx:
    if idx not in all_qubits:
        continue
    x, y = all_qubits[idx]
    is_x = idx in x_anc
    color = '#FF6B6B' if is_x else '#4DA6FF'
    diamond = mpatches.RegularPolygon(
        (x, y), numVertices=4, radius=1.3,
        orientation=np.pi / 4,
        facecolor=color, edgecolor='black', alpha=0.4,
    )
    ax.add_patch(diamond)
    ax.text(x, y, 'X' if is_x else 'Z',
            ha='center', va='center', fontsize=9, fontweight='bold')

for idx in data_idx:
    x, y = all_qubits[idx]
    ax.plot(x, y, 'ko', markersize=7, zorder=5)

coords = list(all_qubits.values())
ax.set_xlim(min(x for x, _ in coords) - 2, max(x for x, _ in coords) + 2)
ax.set_ylim(min(y for _, y in coords) - 2, max(y for _, y in coords) + 2)
ax.set_aspect('equal')
ax.set_title('Rotated surface code (distance 5)')
ax.legend(
    [mpatches.Patch(facecolor='#FF6B6B', alpha=0.4, edgecolor='black'),
     mpatches.Patch(facecolor='#4DA6FF', alpha=0.4, edgecolor='black'),
     plt.Line2D([0], [0], marker='o', color='w',
                markerfacecolor='k', markersize=8)],
    ['X stabilizer (detects Z errors)',
     'Z stabilizer (detects X errors)',
     'Data qubit'],
    loc='upper right',
)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f'{len(data_idx)} data qubits, '
      f'{len(x_anc & measure_idx)} X stabilizers, '
      f'{len(measure_idx - x_anc)} Z stabilizers')

Each data qubit (black dot) sits at the corner of stabilizer patches. Each stabilizer measures the parity of its neighboring data qubits. The checkerboard pattern of X and Z stabilizers means every data qubit is covered by both types — so both X and Z errors can be detected.

The **distance** of the code (5 in this case) is the minimum number of physical errors needed to cause a logical error. Larger distance means more protection, but also more qubits.

In [ ]:
# Error detection on the surface code
# We'll use a simple example: Z stabilizers detecting an X error
# Data qubits: 0-3, Z stabilizer ancillas: 4, 5
surface_error_demo = stim.Circuit('''
    # Inject an X error on data qubit 1
    X_ERROR(1.0) 1

    # Z stabilizer: Z0*Z1 via ancilla 4
    CX 0 4
    CX 1 4
    # Z stabilizer: Z2*Z3 via ancilla 5
    CX 2 5
    CX 3 5
    M 4 5
''')

sampler = surface_error_demo.compile_sampler()
results = sampler.sample(shots=1000)

z_01 = results[:, 0].sum()
z_23 = results[:, 1].sum()
print('With an X error on data qubit 1:')
print(f'  Z stabilizer (q0, q1) triggered: {z_01} / 1000')
print(f'  Z stabilizer (q2, q3) triggered: {z_23} / 1000')

The Z stabilizer covering qubits 0 and 1 fires every time — it detected the X error. The other stays silent. The pattern of triggered stabilizers pinpoints the error location, just like the repetition code but in two dimensions.

In a real surface code, stabilizer measurements are repeated many times, and a classical **decoder** processes the stream of syndromes to identify and correct errors. The surface code can tolerate physical error rates up to about 1% (the **threshold**) while driving the logical error rate exponentially low by increasing the code distance.

### Where things stand

Building a fault-tolerant quantum computer means making the physical error rate low enough to be below threshold, then scaling up the surface code (or a similar code) to achieve the desired logical error rate. This is the central engineering challenge of quantum computing today.

Recent milestones:
- Google demonstrated that increasing surface code size reduces logical error rates — the first time this has been achieved experimentally ([Nature, 2024](https://doi.org/10.1038/s41586-024-08449-y)).
- Current devices operate near the threshold, with physical error rates around 0.1–1%.
- A practical fault-tolerant computer will likely need thousands to millions of physical qubits per logical qubit.

We are at the beginning of the error-corrected era of quantum computing — and that is a very exciting place to be.